# LeetCode #1424: Diagonal Traverse II

https://leetcode.com/problems/diagonal-traverse-ii/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n \log n)$ | $O(n)$ |
| **Optimal: Diagonal Grouping ★** | $O(n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Sort all elements by $(row + col)$ as primary key and $row$ (descending) as secondary key. Sorting adds an $O(n \log n)$ factor where $n$ is the total number of elements.

### Optimal: Diagonal Grouping ★
Use a HashMap keyed by diagonal index $d = row + col$. Append each element to its diagonal bucket in row-first order. Since elements appear in increasing row order, each bucket naturally stores them in ascending-row sequence. Emit all buckets in order of their diagonal key $d = 0, 1, 2, \ldots$ — no sort needed. $O(n)$ time and space.

**Why this is better than Brute Force:** Grouping into buckets avoids the $O(n \log n)$ sort entirely, reducing to a single pass plus one linear output sweep.

**Constraints:**
* $1 \leq nums.length \leq 10^5$
* $1 \leq nums[i].length \leq 10^5$
* $1 \leq nums[i][j] \leq 10^5$
* Total elements $\leq 10^5$


## Solutions

### C#

In [ ]:
public class Solution {
    public int[] FindDiagonalOrder(IList<IList<int>> nums) {
        // Group elements by diagonal key d = row + col
        var diags = new Dictionary<int, List<int>>();
        int maxDiag = 0;
        for (int r = 0; r < nums.Count; r++) {
            for (int c = 0; c < nums[r].Count; c++) {
                int d = r + c;
                if (!diags.ContainsKey(d)) diags[d] = new List<int>();
                diags[d].Add(nums[r][c]); // Rows visited top-to-bottom, so order is correct
                if (d > maxDiag) maxDiag = d;
            }
        }
        // Emit each diagonal in reverse row order (bottom row first per diagonal)
        List<int> result = new List<int>();
        for (int d = 0; d <= maxDiag; d++) {
            if (!diags.ContainsKey(d)) continue;
            List<int> bucket = diags[d];
            for (int i = bucket.Count - 1; i >= 0; i--) {
                result.Add(bucket[i]); // Higher row index comes first in output
            }
        }
        return result.ToArray();
    }
}

### Python

In [ ]:
from collections import defaultdict

class Solution:
    def findDiagonalOrder(self, nums: list[list[int]]) -> list[int]:
        # Group elements by diagonal key d = row + col
        diags: dict[int, list[int]] = defaultdict(list)
        for r, row in enumerate(nums):
            for c, val in enumerate(row):
                diags[r + c].append(val)  # Rows visited top-to-bottom
        result = []
        # Emit each diagonal in reverse row order (bottom row first)
        for d in range(max(diags) + 1):
            result.extend(reversed(diags[d]))
        return result

### Go

In [ ]:
func findDiagonalOrder(nums [][]int) []int {
    // Group elements by diagonal key d = row + col
    diags := make(map[int][]int)
    maxDiag := 0
    for r, row := range nums {
        for c, val := range row {
            d := r + c
            diags[d] = append(diags[d], val) // Rows visited top-to-bottom
            if d > maxDiag {
                maxDiag = d
            }
        }
    }
    result := []int{}
    // Emit each diagonal in reverse row order (bottom row first)
    for d := 0; d <= maxDiag; d++ {
        bucket := diags[d]
        for i := len(bucket) - 1; i >= 0; i-- {
            result = append(result, bucket[i])
        }
    }
    return result
}

### Rust

In [ ]:
use std::collections::HashMap;

impl Solution {
    pub fn find_diagonal_order(nums: Vec<Vec<i32>>) -> Vec<i32> {
        // Group elements by diagonal key d = row + col
        let mut diags: HashMap<usize, Vec<i32>> = HashMap::new();
        let mut max_diag = 0usize;
        for (r, row) in nums.iter().enumerate() {
            for (c, &val) in row.iter().enumerate() {
                let d = r + c;
                diags.entry(d).or_default().push(val); // Rows visited top-to-bottom
                if d > max_diag { max_diag = d; }
            }
        }
        let mut result = Vec::new();
        // Emit each diagonal in reverse row order (bottom row first)
        for d in 0..=max_diag {
            if let Some(bucket) = diags.get(&d) {
                result.extend(bucket.iter().rev());
            }
        }
        result
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `nums = [[1,2,3],[4,5,6],[7,8,9]]`
Diagonals (by $r+c$): d=0:[1], d=1:[2,4], d=2:[3,5,7], d=3:[6,8], d=4:[9]. Reversing each: [1],[4,2],[7,5,3],[8,6],[9]. Result: `[1,4,2,7,5,3,8,6,9]`.

### 2. Slightly Complex
**Input:** `nums = [[1,2,3,4,5]]` (single row)
All elements are on different diagonals (d = 0,1,2,3,4), each bucket has one element. Output is just the row in order: `[1,2,3,4,5]`. Edge case where the matrix has height 1.

### 3. Edge Case: Time Factor
**Input:** Jagged matrix with $10^5$ total elements spread across many rows
Every element is visited exactly once in the build pass and once in the output pass — full $O(n)$ work. Tests maximum total element count.

### 4. Edge Case: Space Factor
**Input:** `nums = [[v] for v in range(1, 100001)]` ($10^5$ single-element rows)
$10^5$ diagonal buckets each holding one element — maximum bucket count. The HashMap holds $O(n)$ entries. Output is just the elements in reverse-row order within each diagonal (each diagonal has one element, so no reversal effect).

### 5. Almost-Impossible but Plausible
**Input:** Long first row (length $5 \times 10^4$) followed by a long second row (same length)
Diagonals span $r+c = 0$ to $r+c \approx 10^5$. Some mid-range diagonals have two elements (one from each row); the reversal step swaps them so the row-1 element precedes the row-0 element. Tests correct bottom-first ordering when rows overlap heavily on the same diagonal.
